# Shaped-Reward Poker REINFORCE on Colab T4

Short experiment for the STAT 4830 final presentation. Tests whether adding a tool-use bonus and fallback penalty breaks the reward-hacking attractor that made Exp B's held-out eval flat at 20% for 100 iterations.

**Starting point:** Exp B's `best_by_eval` LoRA adapter (iter 10, 20% held-out — the checkpoint right before Exp B collapsed into fallback-only behavior).

**Shaped reward:**
```
base_reward  = action_type_match (0.0 or 1.0)
tool_bonus   = 0.3 if model wrote real code else 0.0
             + 0.2 if real code parsed opponent stats else 0.0
penalty      = 0.2 if model used the canned-action fallback else 0.0
reward       = base_reward + tool_bonus - penalty
```

**Run:** 20 iters REINFORCE, batch 4, seed 20260422 (matches Exp B). Held-out eval at iter 0, 5, 10, 15, 20. Total wall clock ~60-90 min on Colab T4.

### How to use

1. **Runtime → Change runtime type → T4 GPU** (required).
2. **Runtime → Run all**.
3. Watch cell 7's streaming output. The decision gates are documented in cell 9.

Keep the tab open the whole time. Colab disconnects after ~90 min of tab inactivity.

## 1. Environment setup

In [1]:
# Colab comes with torch pre-installed. Just add the ML stack.
!pip install -q transformers peft trl datasets accelerate 'bitsandbytes>=0.43' matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00


In [2]:
import os, sys

if not os.path.exists('STAT-4830-RL-project'):
    !git clone --depth 1 https://github.com/aryanarora236/STAT-4830-RL-project.git
os.chdir('STAT-4830-RL-project')
!git pull --ff-only

sys.path.insert(0, os.getcwd())
print('repo at', os.getcwd())

Cloning into 'STAT-4830-RL-project'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 181 (delta 40), reused 160 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 464.84 MiB | 33.13 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Updating files: 100% (288/288), done.
Already up to date.
repo at /content/STAT-4830-RL-project


In [3]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Restart.'
gpu = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu.name}  |  compute {cap[0]}.{cap[1]}  |  {gpu.total_memory / 1e9:.1f} GB VRAM")
print(f"Precision: {'bf16 (Ampere+)' if cap[0] >= 8 else 'fp16 (T4/Volta)'}")

GPU: Tesla T4  |  compute 7.5  |  15.6 GB VRAM
Precision: fp16 (T4/Volta)


## 2. Apply the shaped-reward patch

This modifies `src/poker/training.py` in-memory (on the Colab VM — doesn't affect the repo). Three small insertions: tracking a `was_wrapped` flag per rollout, then adjusting the reward.

If the patch fails, you'll see a clear assertion error — that means the upstream `src/poker/training.py` has drifted and we need to re-find the insertion points.

In [4]:
path = 'src/poker/training.py'
with open(path, 'r') as f:
    src = f.read()

patches = [
    # 1. Initialize was_wrapped at the start of each rollout
    (
        "        for _ in range(self.batch_size):\n            attempted += 1\n            context, question, correct_answer = self.task_generator()",
        "        for _ in range(self.batch_size):\n            attempted += 1\n            was_wrapped = False  # SHAPED REWARD\n            context, question, correct_answer = self.task_generator()",
    ),
    # 2. Flip was_wrapped when we hit the canned-action fallback path
    (
        "                    if attempt_idx == 2:\n                        wrapped_action_code_count += 1\n                        raw_type, raw_amt = parse_action(response_text)",
        "                    if attempt_idx == 2:\n                        wrapped_action_code_count += 1\n                        was_wrapped = True  # SHAPED REWARD\n                        raw_type, raw_amt = parse_action(response_text)",
    ),
    # 3. Compute shaped reward
    (
        "            reward = compute_poker_reward_simple(predicted, correct_answer)\n            if reward > 0:\n                nonzero_reward_count += 1",
        "            base_reward = compute_poker_reward_simple(predicted, correct_answer)\n"
        "            # SHAPED REWARD: reward real code + stat parsing, penalize fallback\n"
        "            tool_bonus = 0.0\n"
        "            if not was_wrapped:\n"
        "                tool_bonus += 0.3\n"
        "            if parsed_stats:\n"
        "                tool_bonus += 0.2\n"
        "            fallback_penalty = 0.2 if was_wrapped else 0.0\n"
        "            reward = base_reward + tool_bonus - fallback_penalty\n"
        "            if reward > 0:\n"
        "                nonzero_reward_count += 1",
    ),
]

for i, (old, new) in enumerate(patches, 1):
    assert old in src, f"Patch {i} could not find its insertion point.\nFirst 160 chars of expected: {old[:160]!r}"
    src = src.replace(old, new, 1)

with open(path, 'w') as f:
    f.write(src)

# Verify
with open(path) as f:
    check = f.read()
for marker in ['was_wrapped = False  # SHAPED REWARD', 'was_wrapped = True  # SHAPED REWARD', 'tool_bonus', 'fallback_penalty']:
    assert marker in check, f'Patch verification failed: missing {marker!r}'

print('All 3 patches applied and verified in src/poker/training.py')

All 3 patches applied and verified in src/poker/training.py


## 3. Load starting checkpoint and run held-out baseline eval

We start from Exp B's `best_by_eval` LoRA adapter. Baseline eval on 20 held-out tasks gives us the number to beat — should come out around 20% matching Exp B's leaderboard.

In [5]:
import shutil, random

START_CKPT_SRC = 'docs/results/poker_rl_expB_mixed20_20260422/poker_rl_expB_mixed20_20260422/best_by_eval'
START_CKPT = 'checkpoints/poker_rl_expB_best'
os.makedirs('checkpoints', exist_ok=True)
if os.path.exists(START_CKPT):
    shutil.rmtree(START_CKPT)
shutil.copytree(START_CKPT_SRC, START_CKPT)
print(f'Starting checkpoint copied to {START_CKPT}')
!ls {START_CKPT}

Starting checkpoint copied to checkpoints/poker_rl_expB_best
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [6]:
# Held-out eval helper — same schema as Aryan's eval_leaderboard.json
from src.training import load_model
from src.poker.agents import PokerLocalLLMAgent
from src.poker.tasks import generate_poker_task
from src.poker.rewards import parse_action

EVAL_SEED = 20260422      # matches Exp B
EVAL_EPISODES = 20

def held_out_eval(model, tokenizer, n_episodes=EVAL_EPISODES, seed=EVAL_SEED):
    """Run n_episodes of held-out eval, return (acc, reward)."""
    random.seed(seed)
    torch.manual_seed(seed)
    agent = PokerLocalLLMAgent(model=model, tokenizer=tokenizer, name='eval_agent', max_steps=3, temperature=0.1)
    correct = 0
    total_reward = 0.0
    for i in range(n_episodes):
        context, question, answer = generate_poker_task()
        pred, _ = agent.run_episode(context, question, answer)
        pred_type = parse_action(pred)[0]
        corr_type = parse_action(answer)[0]
        match = 1.0 if pred_type == corr_type else 0.0
        correct += match
        total_reward += match  # type-match reward
    return correct / n_episodes, total_reward / n_episodes

print('eval helper ready')

eval helper ready


In [ ]:
# Baseline eval on the starting checkpoint
print('Loading starting checkpoint for baseline eval...')
model, tokenizer = load_model(START_CKPT, load_in_4bit=True)
model.eval()

baseline_acc, baseline_reward = held_out_eval(model, tokenizer)
print(f"\n=== BASELINE (iter 0, starting checkpoint) ===")
print(f"Held-out accuracy: {baseline_acc:.1%}")
print(f"Held-out reward:   {baseline_reward:.3f}")
print(f"Expected: ~20% (matches Exp B's best_by_eval leaderboard entry)")

del model, tokenizer
torch.cuda.empty_cache()

Loading starting checkpoint for baseline eval...


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded checkpoint from checkpoints/poker_rl_expB_best
Trainable params: 4,358,144 / 892,974,592 (0.49%)


## 4. Train with shaped reward + eval checkpoints

20 iterations, batch 4. Saves to `checkpoints/poker_rl_shaped/iter_{5,10,15,20}`. Held-out eval runs on each after training.

**What to watch in the streaming output below:**

- Look at `real_code=X/4` in each iteration line. In Exp B this averaged **0.3/4**. Shaping should push it up.
- Training EMA accuracy (`ema=X%`) is noisy and less meaningful than held-out eval.
- **Decision gate:** after `iter_10` eval in the next cell, if held-out accuracy is still 20% or below, consider stopping here. If it moves to 25%+, keep watching.

In [ ]:
# Launch training via CLI so output streams like Aryan's runs did.
# Saves every 5 iters by default.
import os as _os
_os.environ['PYTHONUNBUFFERED'] = '1'

!PYTHONUNBUFFERED=1 python -u scripts/poker_train.py \
    --phase rl \
    --model ./checkpoints/poker_rl_expB_best \
    --seed 20260422 \
    --rl-iterations 20 \
    --rl-batch-size 4 \
    --rl-lr 5e-6 \
    --rl-sample-temperature 0.2 \
    --rl-top-p 0.9 \
    --rl-adv-clip 2.0 \
    --ema-gamma 0.9 \
    --rl-output ./checkpoints/poker_rl_shaped 2>&1 | tee shaped_run.log

## 5. Held-out eval on each saved checkpoint

In [ ]:
import json, glob
from pathlib import Path

results = [{'iteration': 0, 'held_out_acc': baseline_acc, 'held_out_reward': baseline_reward, 'source': 'baseline (Exp B best_by_eval)'}]

ckpt_dirs = sorted(
    glob.glob('checkpoints/poker_rl_shaped/iter_*'),
    key=lambda p: int(p.rsplit('_', 1)[-1])
)
print(f'Found {len(ckpt_dirs)} checkpoints: {[Path(d).name for d in ckpt_dirs]}')

for ckpt in ckpt_dirs:
    it = int(Path(ckpt).name.split('_')[-1])
    print(f'\n--- Evaluating {Path(ckpt).name} ---')
    model, tokenizer = load_model(ckpt, load_in_4bit=True)
    model.eval()
    acc, r = held_out_eval(model, tokenizer)
    print(f'iter {it}: held-out acc = {acc:.1%}, reward = {r:.3f}')
    results.append({'iteration': it, 'held_out_acc': acc, 'held_out_reward': r, 'source': ckpt})
    del model, tokenizer
    torch.cuda.empty_cache()

with open('shaped_eval_leaderboard.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n=== SHAPED-REWARD LEADERBOARD ===')
print(f'{"iter":>5} | {"held-out acc":>12} | {"delta vs baseline":>17}')
for row in results:
    delta = (row['held_out_acc'] - baseline_acc) * 100
    sign = '+' if delta >= 0 else ''
    print(f'{row["iteration"]:>5} | {row["held_out_acc"]:>12.1%} | {sign}{delta:>16.1f} pp')

## 6. Interpret results — what to look for

### Strong positive (A-grade story)
- Baseline: 20%
- Iter 10: ≥30%
- Iter 20: ≥35%
- Trend: monotonic climb
- `real_code` counter: averages ≥2/4 (was 0.3/4 in Exp B)

→ *"Diagnosed reward hacking, shaped the reward to price tool use, held-out generalization improved by X points. Textbook §7."*

### Partial positive
- `real_code` rises to 2-4/4 but held-out eval stays flat ~20-25%

→ *"The intervention fixed the immediate attractor (model writes code again) but didn't translate to better decisions. Deeper problem: reward signal is too sparse for a 1.5B model on a 4-action task."* Still more honest and defensible than the current slides.

### Null result
- Everything flat at 20%

→ The shaped reward didn't help on this timescale. Fall back to current slides. Nothing lost but ~75 min.

### Regression
- Held-out drops below 20%

→ Reward shaping coefficients too aggressive. Worth reporting as "we moved the policy but in the wrong direction."

---

### If you need to kill the run early

`Runtime > Interrupt execution` (or the stop button next to the running cell). Checkpoints saved up to that point are still usable — re-run cell 5 on just those.

## 7. Download results for the slides

Run this cell at the end to download the leaderboard JSON and training log to your laptop.

In [ ]:
from google.colab import files
try:
    files.download('shaped_eval_leaderboard.json')
    files.download('shaped_run.log')
except Exception as e:
    print('download() only works in Colab. Skip if running elsewhere.')
    print(e)